In [288]:
import prepocess, importlib
importlib.reload(prepocess)
from prepocess import preprocess
import pandas as pd


file_path = "syn_data/syn_net_p0.8_mu0.2_1.csv"
node2id = preprocess(file_path=file_path)


50 nodes in the link stream


/Users/acw721/Desktop/research/linkstream/prepocess.py:4: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  link_stream = pd.read_csv(file_path, delimiter=delimiter, names=['source', 'destination', 'timestamp'], index_col=False, skiprows=1)


In [289]:
link_stream = pd.read_csv('preprocessed/syn_net_p0.8_mu0.2_1_pcs.csv')

In [290]:
import torch
import pandas as pd
import numpy as np
from typing import Dict, Tuple, Optional


class OfflineStateManager:
    def __init__(self, num_nodes: int, num_communities: int, device: str = "cpu"):
        self.num_nodes = num_nodes
        self.num_communities = num_communities
        self.device = device

        # -------- static stats (precomputed once) --------
        self.global_degree = torch.zeros(num_nodes, device=device, dtype=torch.float32)
        self.node_lifespans = torch.ones(num_nodes, device=device, dtype=torch.float32)
        self.total_duration = 1.0
        self.m = 0.0

        # -------- time index for delta_t --------
        self.node_time_pos = torch.zeros(num_nodes, dtype=torch.long, device=device)
        self.node_time_indptr: Optional[torch.Tensor] = None
        self.node_time_values: Optional[torch.Tensor] = None

        # -------- dynamic buffers (curr / old) --------

        self.p_src_old_all: Optional[torch.Tensor] = None
        self.p_dst_old_all: Optional[torch.Tensor] = None
        self.num_instances_cached: int = 0
        # -------- other dynamic state --------
        self.p_prev: Dict[int, torch.Tensor] = {}

        # 数值稳定用的小 eps
        self.eps = 1e-12

    # ---------------------- one-time preprocessing ----------------------
    def pre_scan_data(self, link_stream: pd.DataFrame):
        self.m = float(len(link_stream))
        print(f"Total links: {int(self.m)}")

        # ---- global_degree k_u ----
        all_nodes = pd.concat([link_stream["source"], link_stream["destination"]], ignore_index=True)
        all_nodes_t = torch.tensor(all_nodes.values, dtype=torch.long, device=self.device)
        self.global_degree = torch.bincount(all_nodes_t, minlength=self.num_nodes).float()

        # ---- total duration ----
        t_max = link_stream["timestamp"].max()
        t_min = link_stream["timestamp"].min()
        total_duration = t_max - t_min
        self.total_duration = float(total_duration) if total_duration > 0 else 1.0
        print(f"T_duration = {total_duration}")

        # ---- node lifespans (optional) ----
        sources = link_stream[["source", "timestamp"]].rename(columns={"source": "node"})
        destinations = link_stream[["destination", "timestamp"]].rename(columns={"destination": "node"})
        all_events = pd.concat([sources, destinations], ignore_index=True)

        node_stats = all_events.groupby("node")["timestamp"].agg(["min", "max"])
        epsilon = 1.0
        lifespans = (node_stats["max"] - node_stats["min"]).clip(lower=epsilon)

        self.node_lifespans.fill_(epsilon)
        active_ids = torch.tensor(node_stats.index.values, device=self.device, dtype=torch.long)
        active_vals = torch.tensor(lifespans.values, device=self.device, dtype=torch.float32)
        self.node_lifespans[active_ids] = active_vals
        print(f"Max Lifespan: {self.node_lifespans.max().item()}")

        # ---- build node_time_values: timestamps sorted by (node, timestamp) ----
        all_events_sorted = all_events.sort_values(["node", "timestamp"], kind="mergesort")
        nodes_np = all_events_sorted["node"].to_numpy(dtype=np.int64, copy=False)
        times_np = all_events_sorted["timestamp"].to_numpy(copy=False)

        if np.issubdtype(times_np.dtype, np.integer):
            if times_np.min() >= np.iinfo(np.int32).min and times_np.max() <= np.iinfo(np.int32).max:
                times_np = times_np.astype(np.int32, copy=False)
            else:
                times_np = times_np.astype(np.int64, copy=False)
        else:
            times_np = times_np.astype(np.float32, copy=False)

        counts = np.bincount(nodes_np, minlength=self.num_nodes).astype(np.int64)
        indptr = np.empty(self.num_nodes + 1, dtype=np.int64)
        indptr[0] = 0
        np.cumsum(counts, out=indptr[1:])

        self.node_time_indptr = torch.from_numpy(indptr).to(self.device)
        self.node_time_values = torch.from_numpy(times_np).to(self.device)

        print(f"Total node appearances stored: {self.node_time_values.numel()} (expected ~ {2*int(self.m)})")

        # epoch0 init
        self.reset_time_pos()

    # ---------------------- epoch / pass control ----------------------
    def reset_time_pos(self):
        """Reset per-node appearance pointer (for delta_t)."""
        self.node_time_pos.zero_()

    def reset_p_prev(self):
        self.p_prev.clear()

    # ---------------------- delta_t computation ----------------------
    def get_delta_for_batch(
        self,
        src: torch.Tensor,  # [B] long
        dst: torch.Tensor,  # [B] long
    ) -> Tuple[torch.Tensor, torch.Tensor, Dict[int, int]]:
        """
        Compute delta_src/delta_dst for each occurrence in this batch.
        Uses node_time_pos + temp_pos to handle repeated nodes within batch.

        Returns:
          delta_src, delta_dst: [B] float32
          occ_counts: {u: count_in_batch} for advancing node_time_pos
        """
        assert self.node_time_indptr is not None and self.node_time_values is not None, \
            "Call pre_scan_data() first to build node_time_indptr/node_time_values."

        assert src.dtype == torch.long and dst.dtype == torch.long
        B = src.numel()
        device = self.device

        delta_src = torch.zeros(B, device=device, dtype=torch.float32)
        delta_dst = torch.zeros(B, device=device, dtype=torch.float32)

        temp_pos: Dict[int, int] = {}
        occ_counts: Dict[int, int] = {}

        def _next_delta(u: int) -> float:
            base_pos = int(self.node_time_pos[u].item())
            offset = temp_pos.get(u, 0)
            pos = base_pos + offset

            start = int(self.node_time_indptr[u].item())
            end = int(self.node_time_indptr[u + 1].item())
            idx = start + pos

            temp_pos[u] = offset + 1
            occ_counts[u] = occ_counts.get(u, 0) + 1

            # last appearance => fallback
            if idx + 1 >= end:
                return 1.0

            t1 = self.node_time_values[idx]
            t2 = self.node_time_values[idx + 1]
            dt = (t2 - t1).float()
            dt = torch.clamp(dt, min=1.0)  # avoid 0/negative
            return float(dt.item())

        for i in range(B):
            u = int(src[i].item())
            v = int(dst[i].item())
            delta_src[i] = _next_delta(u)
            delta_dst[i] = _next_delta(v)

        return delta_src, delta_dst, occ_counts

    # ---------------------- commit after each batch ----------------------
    @torch.no_grad()
    def commit_batch(self, last_p_nodes: Dict[int, torch.Tensor], occ_counts: Dict[int, int]):
        for u, p_last in last_p_nodes.items():
            self.p_prev[int(u)] = p_last.detach().clone()

        for u, cnt in occ_counts.items():
            self.node_time_pos[int(u)] += int(cnt)


    @torch.no_grad()
    def init_old_p_cache(
        self,
        num_instances: int,
        dtype: torch.dtype = torch.float16,
        pin_memory: bool = False,
        init_uniform: bool = True,
    ):
        K = self.num_communities
        self.num_instances_cached = int(num_instances)

        if init_uniform:
            val = 1.0 / float(K)
            p_src = torch.full((num_instances, K), val, dtype=dtype, device="cpu")
            p_dst = torch.full((num_instances, K), val, dtype=dtype, device="cpu")
        else:
            p_src = torch.zeros((num_instances, K), dtype=dtype, device="cpu")
            p_dst = torch.zeros((num_instances, K), dtype=dtype, device="cpu")

        pin_ok = pin_memory and torch.cuda.is_available()
        if pin_ok:
            p_src = p_src.pin_memory()
            p_dst = p_dst.pin_memory()

        self.p_src_old_all = p_src
        self.p_dst_old_all = p_dst


    @torch.no_grad()
    def write_old_p_batch(self, start_idx: int, end_idx: int, 
                        p_src: torch.Tensor, p_dst: torch.Tensor):
        if self.p_src_old_all is None or self.p_dst_old_all is None:
            raise RuntimeError("old p cache not initialized.")
        

        p_src_copy = p_src.clone().detach().cpu().to(dtype=self.p_src_old_all.dtype)
        p_dst_copy = p_dst.clone().detach().cpu().to(dtype=self.p_dst_old_all.dtype)
        
        self.p_src_old_all[start_idx:end_idx].copy_(p_src_copy)
        self.p_dst_old_all[start_idx:end_idx].copy_(p_dst_copy)


    def read_old_p_batch(
        self,
        start_idx: int,
        end_idx: int,
        device: torch.device,
        dtype: torch.dtype,
        non_blocking: bool = True,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Called in training pass:
        Fetch old p for this batch and move to GPU (or target device).
        Returns p_src_old, p_dst_old: [B,K] on `device` with `dtype`.
        """
        if self.p_src_old_all is None or self.p_dst_old_all is None:
            raise RuntimeError("old p cache not initialized. Call init_old_p_cache(num_instances) first.")

        p_src_old = self.p_src_old_all[start_idx:end_idx].to(device=device, dtype=dtype, non_blocking=non_blocking)
        p_dst_old = self.p_dst_old_all[start_idx:end_idx].to(device=device, dtype=dtype, non_blocking=non_blocking)
        return p_src_old, p_dst_old

In [430]:
device = 'cuda' if torch.cuda.is_available() else 'mps'
node_set = set(pd.concat([link_stream['source'], link_stream['destination']], ignore_index=True))
num_nodes = len(node_set)
print(f'num_nodes = {num_nodes} ')
num_comms = 5
state_mgr = OfflineStateManager(num_nodes, num_comms, device=device)
state_mgr.pre_scan_data(link_stream)
state_mgr.init_old_p_cache(len(link_stream), dtype=torch.float16, pin_memory=True, init_uniform=True)

num_nodes = 50 
Total links: 469
T_duration = 2830
Max Lifespan: 2808.0
Total node appearances stored: 938 (expected ~ 938)


In [431]:
from tgn.utils.my_data import get_data  
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

data = 'syn_net_p0.8_mu0.2_pcs'

node_feat, edge_feat, full_data = get_data(data, "preprocessed/syn_net_p0.8_mu0.2_1_pcs.csv", node_embedding_method="one-hot")
max_idx = max(full_data.unique_nodes)

cannot find (./data/syn_net_p0.8_mu0.2_pcs.npy), use zero-vector for edge feat (dim=16)...
Use one-hot init for node embedding. 
The dataset has 469 interactions, involving 50 different nodes


In [294]:
import math
import logging
import time
import sys
import argparse
import numpy as np
import pickle
from pathlib import Path


NUM_HEADS = 4
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.0001
NODE_DIM = node_feat.shape[1]
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = NODE_DIM
device = 'cuda' if torch.cuda.is_available() else 'mps'

prefix = 'max_lmod'
data='link_stream'

Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


In [60]:
'''
BATCH_SIZE = 200
num_instance = len(full_data.sources)

state_mgr.reset_time_pos()
for k in range(1, 2):
    
    start_idx = BATCH_SIZE * k
    end_idx = min(num_instance, BATCH_SIZE * (k + 1))

    sources_batch = full_data.sources[start_idx:end_idx]
    destinations_batch = full_data.destinations[start_idx:end_idx]
    timestamps_batch = full_data.timestamps[start_idx:end_idx]
    edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]
    
    src = torch.as_tensor(sources_batch, dtype=torch.long, device=device)
    dst = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)
    ts  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)
    print("src:", src)
    delta_src, delta_dst, occ_counts = state_mgr.get_delta_for_batch(src, dst)
    print("delta_src:", delta_src)
    print("occ count", occ_counts)
'''

'\nBATCH_SIZE = 200\nnum_instance = len(full_data.sources)\n\nstate_mgr.reset_time_pos()\nfor k in range(1, 2):\n\n    start_idx = BATCH_SIZE * k\n    end_idx = min(num_instance, BATCH_SIZE * (k + 1))\n\n    sources_batch = full_data.sources[start_idx:end_idx]\n    destinations_batch = full_data.destinations[start_idx:end_idx]\n    timestamps_batch = full_data.timestamps[start_idx:end_idx]\n    edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]\n\n    src = torch.as_tensor(sources_batch, dtype=torch.long, device=device)\n    dst = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)\n    ts  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)\n    print("src:", src)\n    delta_src, delta_dst, occ_counts = state_mgr.get_delta_for_batch(src, dst)\n    print("delta_src:", delta_src)\n    print("occ count", occ_counts)\n'

In [295]:
from tgn.utils.utils import EarlyStopMonitor, get_neighbor_finder, MLP

ngh_finder = get_neighbor_finder(full_data, uniform=True, max_node_idx=max_idx)

In [296]:
from tgn.utils.my_data import get_data, compute_time_statistics, compute_time_statistics_undirected
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

mean_time_shift, std_time_shift= compute_time_statistics_undirected(full_data.sources, 
                                full_data.destinations, 
                                full_data.timestamps)


In [444]:
from tgn.model.my_tgn import TGN
importlib.reload(sys.modules['tgn.model.my_tgn'])
from pathlib import Path


NUM_NEIGHBORS = 20
NUM_HEADS = 4
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.0005
NODE_DIM = node_feat.shape[1]
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = NODE_DIM
num_communities = 5

device = 'cuda' if torch.cuda.is_available() else 'mps'
aggregator = 'last'
memory_update_at_end = False
embedding_module = 'graph_attention'
# graph_attention, graph_sum, time, identity
message_function = 'mlp'

prefix = 'syn_net'

tgn = TGN(
    neighbor_finder=ngh_finder,
    node_features=node_feat,
    edge_features=edge_feat,
    device=device,
    n_layers=NUM_LAYER,
    n_heads=NUM_HEADS,
    dropout=DROP_OUT,
    use_memory=USE_MEMORY,
    message_dimension=MESSAGE_DIM,
    memory_dimension=MEMORY_DIM,
    n_neighbors=NUM_NEIGHBORS,
    mean_time_shift=mean_time_shift,
    std_time_shift=std_time_shift,
    use_destination_embedding_in_message=True,
    use_source_embedding_in_message=True,
    memory_update_at_start=not memory_update_at_end,
    embedding_module_type=embedding_module,
    message_function=message_function,
    aggregator_type=aggregator, 
    
    num_communities=num_communities
).to(device)

optimizer = torch.optim.Adam(tgn.parameters(), lr=LEARNING_RATE)

In [391]:
import math 
import logging
import time


BATCH_SIZE = 500

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


num_instance = len(full_data.sources)
num_batches = math.ceil(len(full_data.sources) / BATCH_SIZE)
logger.debug(f'num_batches: {num_batches}')

DEBUG:root:num_batches: 1


In [299]:
def build_batch_updates(src, dst,
                        p_src_curr, p_dst_curr):
    last_p_nodes = {}

    def _acc(node_ids, p_curr):
        for i in range(node_ids.numel()):
            u = int(node_ids[i].item())
            last_p_nodes[u] = p_curr[i]

    _acc(src, p_src_curr)
    _acc(dst, p_dst_curr)
    return last_p_nodes

In [194]:
def print_grads(tag: str):
    print(f"\n=== Batch {k+1} {tag} Gradients ===")
    for name, param in tgn.named_parameters():
        if param.grad is not None:
            g = param.grad
            print(f"{name}: norm={g.norm().item():.6f}, max={g.abs().max().item():.6f}")


In [ ]:
import importlib, sys, time
import tgn.model.loss as loss_mod
importlib.reload(loss_mod)
longitudinal_modularity_loss = loss_mod.longitudinal_modularity_loss
NUM_EPOCHS = 50
bp_every = 2

for epoch in range(NUM_EPOCHS):
    tgn = tgn.train()
    start_epoch_time = time.time()

    if USE_MEMORY:
        tgn.memory.__init_memory__()

    state_mgr.reset_time_pos()
    state_mgr.reset_p_prev()
    tgn.set_neighbor_finder(ngh_finder)

    epoch_loss = 0.0
    epoch_null_loss = 0.0
    epoch_collapse_loss = 0.0
    epoch_steps = 0

    logger.info(f"Starting epoch {epoch+1} / {NUM_EPOCHS}")

    # 关键：epoch 开头清一次梯度
    optimizer.zero_grad(set_to_none=True)

    for k in range(num_batches):
        print(f"Processing batch {k+1} / {num_batches}")
        start_idx = BATCH_SIZE * k
        end_idx = min(num_instance, BATCH_SIZE * (k + 1))

        sources_batch = full_data.sources[start_idx:end_idx]
        destinations_batch = full_data.destinations[start_idx:end_idx]
        timestamps_batch = full_data.timestamps[start_idx:end_idx]
        edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]

        src = torch.as_tensor(sources_batch, dtype=torch.long, device=device)
        dst = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)
        ts  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)

        delta_src, delta_dst, occ_counts = state_mgr.get_delta_for_batch(src, dst)

        p_src, p_dst = tgn.compute_community_prob(
            sources_batch, destinations_batch, timestamps_batch, edge_idxs_batch
        )

        p_src_old, p_dst_old = state_mgr.read_old_p_batch(
            start_idx, end_idx,
            device=device,
            dtype=p_src.dtype,
            non_blocking=False
        )

        if not torch.isfinite(p_src).all():
            raise RuntimeError("p_src NaN right after forward")
        if not torch.isfinite(p_dst).all():
            raise RuntimeError("p_dst NaN right after forward")

        last_p_nodes = build_batch_updates(src, dst, p_src, p_dst)

        # loss
        loss= longitudinal_modularity_loss(
            p_src, p_dst,
            src, dst,
            state_mgr.node_lifespans, state_mgr.global_degree,
            state_mgr.m, state_mgr.total_duration,
            state_mgr.p_prev,
            delta_src, delta_dst,
            null_weight=1.0,
            csc_weight=0.0,
            collapse_weight=1.0,
            csc_norm="l2",
            neg_weight=1.0
        )

        # 关键：缩放后再 backward（避免累计后更新幅度变大）
        (loss / bp_every).backward()

        # 每个 batch 都更新“状态”（不影响梯度累计）
        state_mgr.commit_batch(last_p_nodes, occ_counts)

        if USE_MEMORY:
            tgn.memory.detach_memory()

        # 记录日志（这里用原始 loss，不要用除过 bp_every 的）
        epoch_loss += float(loss.detach().item())
        #epoch_null_loss += float(last_components[0].item())
        #epoch_collapse_loss += float(last_components[1].item())
        epoch_steps += 1

        # 关键：每 bp_every 步更新一次参数
        do_step = ((k + 1) % bp_every == 0)
        last_batch = (k + 1 == num_batches)

        if do_step or last_batch:
            torch.nn.utils.clip_grad_norm_(tgn.parameters(), 1.0)

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

    if epoch_steps > 0:
        mean_loss = epoch_loss / epoch_steps
        #mean_null = epoch_null_loss / epoch_steps
        #mean_collapse = epoch_collapse_loss / epoch_steps
        logger.info(
            f"[Epoch {epoch}] mean loss={mean_loss:.6f} | "
            #f"mod={mean_null:.6f}, collapse loss {mean_collapse:.6f}"
        )

    if USE_MEMORY:
        memory_backup = tgn.memory.backup_memory()
        tgn.memory.restore_memory(memory_backup)

INFO:root:Starting epoch 1 / 50


Processing batch 1 / 1
E: tensor(0.0081, device='mps:0', grad_fn=<DivBackward0>)
A: tensor(0.3746, device='mps:0', grad_fn=<MeanBackward0>)
collapse: tensor(0.0027, device='mps:0', grad_fn=<ClampBackward1>)
neg: tensor(0.2003, device='mps:0', grad_fn=<MulBackward0>)


INFO:root:[Epoch 0] mean loss=-0.303752 | 
INFO:root:Starting epoch 2 / 50


Processing batch 1 / 1


INFO:root:[Epoch 1] mean loss=-0.315277 | 
INFO:root:Starting epoch 3 / 50


...

E: tensor(0.0082, device='mps:0', grad_fn=<DivBackward0>)
A: tensor(0.6266, device='mps:0', grad_fn=<MeanBackward0>)
collapse: tensor(0.0202, device='mps:0', grad_fn=<ClampBackward1>)
neg: tensor(0.2077, device='mps:0', grad_fn=<MulBackward0>)
Processing batch 1 / 1


INFO:root:[Epoch 48] mean loss=-0.580170 | 
INFO:root:Starting epoch 50 / 50


E: tensor(0.0086, device='mps:0', grad_fn=<DivBackward0>)
A: tensor(0.6785, device='mps:0', grad_fn=<MeanBackward0>)
collapse: tensor(0.0259, device='mps:0', grad_fn=<ClampBackward1>)
neg: tensor(0.2127, device='mps:0', grad_fn=<MulBackward0>)
Processing batch 1 / 1


INFO:root:[Epoch 49] mean loss=-0.476513 | 


E: tensor(0.0115, device='mps:0', grad_fn=<DivBackward0>)
A: tensor(0.7189, device='mps:0', grad_fn=<MeanBackward0>)
collapse: tensor(0.1528, device='mps:0', grad_fn=<ClampBackward1>)
neg: tensor(0.2602, device='mps:0', grad_fn=<MulBackward0>)


In [448]:
import json
import numpy as np
import pandas as pd
import torch

def export_probs_to_csv(
    tgn,
    full_data,
    BATCH_SIZE: int,
    out_csv_path: str,
    id2node: dict
):
    tgn.eval()
    tgn.memory.__init_memory__()
    num_instance = len(full_data.sources)
    rows = []

    def _to_py_int(x):
        if isinstance(x, np.generic):
            return x.item()
        if torch.is_tensor(x):
            return x.item()
        return x

    def _map_id_to_node_name(x):
        x = _to_py_int(x)
        return id2node[int(x)]

    with torch.no_grad():
        for k in range((num_instance + BATCH_SIZE - 1) // BATCH_SIZE):
            start_idx = BATCH_SIZE * k
            end_idx = min(num_instance, BATCH_SIZE * (k + 1))

            sources_batch = full_data.sources[start_idx:end_idx]
            destinations_batch = full_data.destinations[start_idx:end_idx]
            timestamps_batch = full_data.timestamps[start_idx:end_idx]
            edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]

            p_src, p_dst = tgn.compute_community_prob(
                sources_batch, destinations_batch, timestamps_batch, edge_idxs_batch
            )

            p_src_np = p_src.detach().cpu().float().numpy()
            p_dst_np = p_dst.detach().cpu().float().numpy()

            src_commu = p_src_np.argmax(axis=1).astype(np.int64)
            dst_commu = p_dst_np.argmax(axis=1).astype(np.int64)

            for i in range(end_idx - start_idx):
                rows.append({
                    "source": _map_id_to_node_name(sources_batch[i]),
                    "destination": _map_id_to_node_name(destinations_batch[i]),
                    "timestamp": float(_to_py_int(timestamps_batch[i])),
                    "p_src": json.dumps(p_src_np[i].tolist()),
                    "p_dst": json.dumps(p_dst_np[i].tolist()),
                    "source_commu": int(src_commu[i]),
                    "destination_commu": int(dst_commu[i]),
                })

    df = pd.DataFrame(rows, columns=[
        "source", "destination", "timestamp",
        "p_src", "p_dst", "source_commu", "destination_commu"
    ])
    df.to_csv(out_csv_path, index=False)
    print(f"Saved: {out_csv_path}  (rows={len(df)})")
id2node = {idx: node for node, idx in node2id.items()}


export_probs_to_csv(tgn, full_data, BATCH_SIZE, "result/TGN_community.csv", id2node=id2node)

Saved: result/TGN_community.csv  (rows=469)
